# ShadowMesh Cyber Deception — Dataset Exploration & Anomaly Detection

**ShadowMesh** is an AI-driven active cyber deception platform. This notebook explores the real attack behavioral dataset captured by running Kali Linux tools (nmap, hydra, sqlmap, OWASP payloads) against live Docker honeypot containers.

### What you'll find here
1. Dataset overview and feature distributions
2. Benign vs attack behavioral signatures visualized
3. IsolationForest anomaly detection — loaded and used live
4. Per-attack-phase behavioral fingerprints
5. Feature importance and separability analysis

---
**Companion model:** [ShadowMesh Anomaly Detector + RL Topology Optimizer](https://www.kaggle.com/models/deaker09/shadowmesh-anomaly-rl-models)  
**Source code:** [github.com/Deaker-09/ShadowMess-AI](https://github.com/Deaker-09/ShadowMess-AI)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams['figure.facecolor'] = '#0d0d0d'
plt.rcParams['axes.facecolor'] = '#1a1a1a'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'
plt.rcParams['grid.color'] = '#333333'
plt.rcParams['grid.alpha'] = 0.5

SHADOW_RED    = '#E24B4A'
SHADOW_GREEN  = '#1D9E75'
SHADOW_AMBER  = '#EF9F27'
SHADOW_PURPLE = '#7F77DD'
SHADOW_BLUE   = '#378ADD'

print('Libraries loaded')

In [ ]:
import os, glob

def find_input(filename):
    """Finds a file anywhere under /kaggle/input regardless of dataset slug."""
    matches = glob.glob(f'/kaggle/input/**/{filename}', recursive=True)
    if matches:
        return matches[0]
    # Local fallback for running outside Kaggle
    local = os.path.join(os.getcwd(), filename)
    if os.path.exists(local):
        return local
    raise FileNotFoundError(f'{filename} not found in /kaggle/input or local directory')

DATASET_CSV   = find_input('real_attack_dataset.csv')
QTABLE_CSV    = find_input('q_table_readable.csv')
MODEL_PKL     = find_input('isolation_forest_model.pkl')
QTABLE_NPY    = find_input('q_table.npy')

print(f'Dataset CSV : {DATASET_CSV}')
print(f'Q-table CSV : {QTABLE_CSV}')
print(f'Model PKL   : {MODEL_PKL}')
print(f'Q-table NPY : {QTABLE_NPY}')


## 1. Load Dataset

In [ ]:
df = pd.read_csv(DATASET_CSV)

FEATURES = [
    'timing_delta_ms', 'port_entropy', 'unique_ports_5s',
    'login_rate_60s', 'command_diversity', 'lateral_spread', 'action_type_encoded'
]

print(f'Dataset shape: {df.shape}')
print(f'\nClass distribution:')
print(df['label'].value_counts().rename({0: 'benign', 1: 'attack'}))
print(f'\nAttack phases:')
print(df[df['label']==1]['attack_phase'].value_counts().to_string())
df.head(3)

## 2. Feature Distributions — Benign vs Attack

In [ ]:
benign = df[df['label'] == 0]
attack = df[df['label'] == 1]

FEATURE_LABELS = {
    'timing_delta_ms': 'Timing Delta (ms)\nLow = automated/bursty',
    'port_entropy': 'Port Entropy\nHigh = broad port sweep',
    'unique_ports_5s': 'Unique Ports / 5s\nHigh = rapid scanning',
    'login_rate_60s': 'Login Rate / 60s\nHigh = brute force',
    'command_diversity': 'Command Diversity\nHigh = advanced attacker',
    'lateral_spread': 'Lateral Spread\nHigh = pivoting',
    'action_type_encoded': 'Action Type\n0=scan 1=login 2=cmd 3=data 4=lateral 5=cred',
}

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.suptitle('Feature Distributions: Benign vs Real Attack Traffic', 
             fontsize=16, fontweight='bold', color='white', y=1.01)
axes = axes.flatten()

for i, feat in enumerate(FEATURES):
    ax = axes[i]
    ax.hist(benign[feat], bins=40, alpha=0.6, color=SHADOW_GREEN,  label='Benign',  density=True)
    ax.hist(attack[feat],  bins=20, alpha=0.8, color=SHADOW_RED,   label='Attack',  density=True)
    ax.set_title(FEATURE_LABELS[feat], fontsize=9, color='white')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].set_visible(False)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print('Key insight: timing_delta_ms and port_entropy are the strongest separators')

## 3. Attack Phase Behavioral Fingerprints

In [ ]:
# Normalize each feature to [0,1] for radar-style comparison
phase_df = attack.groupby('attack_phase')[FEATURES].mean()

# Min-max normalize using full dataset range
feat_min = df[FEATURES].min()
feat_max = df[FEATURES].max()
phase_norm = (phase_df - feat_min) / (feat_max - feat_min + 1e-9)

fig, ax = plt.subplots(figsize=(16, 7))

cmap = plt.cm.get_cmap('tab20', len(phase_norm))
x = np.arange(len(FEATURES))
width = 0.06
n = len(phase_norm)

for i, (phase, row) in enumerate(phase_norm.iterrows()):
    offset = (i - n/2) * width
    bars = ax.bar(x + offset, row.values, width, 
                  label=phase, color=cmap(i), alpha=0.85, edgecolor='none')

ax.set_xticks(x)
short_labels = ['timing\n(ms)', 'port\nentropy', 'ports\n/5s', 'logins\n/60s', 'cmd\ndiversity', 'lateral\nspread', 'action\ntype']
ax.set_xticklabels(short_labels, fontsize=10)
ax.set_ylabel('Normalized Feature Value (0–1)', fontsize=11)
ax.set_title('Attack Phase Behavioral Fingerprints\n(each bar = normalized mean feature value)', 
             fontsize=13, fontweight='bold')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8, framealpha=0.3)
ax.grid(True, axis='y', alpha=0.3)
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig('attack_phase_fingerprints.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()

## 4. Correlation Heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, subset, title, color in [
    (axes[0], benign, 'Benign Traffic Correlations', SHADOW_GREEN),
    (axes[1], attack, 'Attack Traffic Correlations', SHADOW_RED),
]:
    corr = subset[FEATURES].corr()
    short = ['timing', 'entropy', 'ports_5s', 'login_r', 'cmd_div', 'lateral', 'act_type']
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(
        corr, ax=ax, mask=mask,
        cmap='RdYlGn', center=0, vmin=-1, vmax=1,
        annot=True, fmt='.2f', annot_kws={'size': 9},
        xticklabels=short, yticklabels=short,
        linewidths=0.5, linecolor='#333',
        cbar_kws={'shrink': 0.8}
    )
    ax.set_title(title, fontsize=12, fontweight='bold', color=color, pad=12)

plt.suptitle('Feature Correlations — Attack vs Benign Traffic', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('correlation_heatmaps.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print('Attack traffic shows stronger lateral_spread ↔ command_diversity correlation')

## 5. Load the Pre-Trained IsolationForest and Score Live

In [ ]:
model = joblib.load(MODEL_PKL)
print('IsolationForest loaded')
print(f'  Estimators : {model.n_estimators}')
print(f'  Contamination: {model.contamination}')

X = df[FEATURES].values.astype(np.float32)
y = df['label'].values

preds = model.predict(X)          # -1 = anomaly, 1 = normal
scores = model.decision_function(X)
threat_scores = np.clip(1.0 - (scores + 0.5), 0.0, 1.0)

df['threat_score'] = threat_scores
df['predicted_anomaly'] = (preds == -1).astype(int)

benign_acc  = (preds[y == 0] ==  1).mean()
attack_rec  = (preds[y == 1] == -1).mean()
precision   = df[df['predicted_anomaly']==1]['label'].mean() if df['predicted_anomaly'].sum() > 0 else 0

print(f'\nModel Performance on Real Data:')
print(f'  Benign accuracy : {benign_acc:.1%}')
print(f'  Attack recall   : {attack_rec:.1%}')
print(f'  Anomaly precision: {precision:.1%}')

## 6. Threat Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: threat score distributions
ax = axes[0]
ax.hist(df[df['label']==0]['threat_score'], bins=50, alpha=0.7, 
        color=SHADOW_GREEN, label=f'Benign (n={len(benign):,})', density=True)
ax.hist(df[df['label']==1]['threat_score'], bins=20, alpha=0.8, 
        color=SHADOW_RED, label=f'Attack (n={len(attack)})', density=True)
ax.axvline(0.5, color=SHADOW_AMBER, linestyle='--', linewidth=2, label='Threshold (0.5)')
ax.set_xlabel('Threat Score', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Threat Score Distribution\n(IsolationForest output)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Right: per-phase mean threat score
ax2 = axes[1]
phase_scores = df.groupby('attack_phase')['threat_score'].mean().sort_values(ascending=True)
colors = [SHADOW_GREEN if p == 'benign' else SHADOW_RED for p in phase_scores.index]
bars = ax2.barh(phase_scores.index, phase_scores.values, color=colors, alpha=0.85, edgecolor='none')
ax2.axvline(0.5, color=SHADOW_AMBER, linestyle='--', linewidth=1.5, label='Threshold')
ax2.set_xlabel('Mean Threat Score', fontsize=12)
ax2.set_title('Mean Threat Score per Attack Phase', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, axis='x', alpha=0.3)

for bar, val in zip(bars, phase_scores.values):
    ax2.text(val + 0.01, bar.get_y() + bar.get_height()/2, 
             f'{val:.2f}', va='center', fontsize=8, color='white')

plt.suptitle('IsolationForest Anomaly Scoring — Real Honeypot Data', 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('threat_scores.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()

## 7. Score a New Action Live

In [ ]:
# Feature order: timing_delta_ms, port_entropy, unique_ports_5s,
#                login_rate_60s, command_diversity, lateral_spread, action_type_encoded

test_cases = [
    ('Normal user activity',     [2500.0, 0.2, 1.0,  0.0, 2.0, 1.0, 2.0]),
    ('nmap port scan',           [  45.0, 3.8, 22.0, 0.0, 1.0, 4.0, 0.0]),
    ('Hydra SSH brute force',    [ 120.0, 0.1,  1.0, 18.0, 1.0, 1.0, 1.0]),
    ('Lateral movement + creds', [ 200.0, 0.9,  5.0,  2.0, 5.0, 8.0, 4.0]),
    ('Ransomware burst',         [  15.0, 2.5, 12.0,  0.0, 6.0, 5.0, 2.0]),
]

print(f'{"Action":<30} {"Threat Score":>12} {"Anomalous?":>12}')
print('-' * 56)

for name, features in test_cases:
    X_test = np.array([features], dtype=np.float32)
    raw = model.decision_function(X_test)[0]
    ts = float(np.clip(1.0 - (raw + 0.5), 0.0, 1.0))
    pred = model.predict(X_test)[0]
    flag = 'ANOMALOUS' if pred == -1 else 'normal'
    print(f'{name:<30} {ts:>12.3f} {flag:>12}')

## 8. t-SNE Visualization — Benign vs Attack Clusters

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# Sample for speed
sample_b = benign.sample(n=500, random_state=42)
sample_a = attack.copy()
sample = pd.concat([sample_b, sample_a])

X_scaled = StandardScaler().fit_transform(sample[FEATURES])
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
emb = tsne.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(12, 8))

# Benign cluster
mask_b = sample['label'].values == 0
ax.scatter(emb[mask_b, 0], emb[mask_b, 1], 
           c=SHADOW_GREEN, alpha=0.3, s=15, label='Benign')

# Attack points — colored by phase
phases = sample[sample['label']==1]['attack_phase'].values
unique_phases = sorted(set(phases))
phase_colors = plt.cm.tab20(np.linspace(0, 1, len(unique_phases)))
phase_color_map = dict(zip(unique_phases, phase_colors))

mask_a = sample['label'].values == 1
attack_emb = emb[mask_a]
for p, color in phase_color_map.items():
    idx = [i for i, ph in enumerate(phases) if ph == p]
    ax.scatter(attack_emb[idx, 0], attack_emb[idx, 1],
               c=[color], s=80, zorder=5, edgecolors='white', linewidths=0.5, label=p)

ax.set_title('t-SNE: Benign vs Attack Behavioral Clusters\n(Real honeypot data — each attack phase = distinct marker)', 
             fontsize=13, fontweight='bold')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8, framealpha=0.3)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('tsne_clusters.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print('Attack phases cluster away from benign traffic — strong behavioral separability')

## 9. Statistical Summary

In [ ]:
print('=== BENIGN TRAFFIC ===')
print(benign[FEATURES].describe().round(3).to_string())
print('\n=== ATTACK TRAFFIC ===')
print(attack[FEATURES].describe().round(3).to_string())

print('\n=== KEY BEHAVIORAL DIFFERENCES (attack mean / benign mean) ===')
ratio = attack[FEATURES].mean() / (benign[FEATURES].mean() + 1e-9)
for feat, r in ratio.sort_values(ascending=False).items():
    bar = '#' * min(int(r * 3), 40)
    print(f'  {feat:<25} {r:>6.1f}x  {bar}')

## 10. How This Data Was Captured

```
Kali Linux WSL
     │
     ├── nmap -sV -sC      → service fingerprint (Redis, Uvicorn, Neo4j, Gunicorn)
     ├── hydra              → 165 SSH attempts (11 users × 15 passwords)
     ├── sqlmap             → SQL injection probes on REST API
     ├── curl (OWASP)       → ShellShock, Log4Shell, SSRF, JWT bypass, SQLi
     ├── dig                → DNS queries to honeypot DNS server
     └── ldapsearch         → Active Directory enumeration
           │
           ▼
     ShadowMesh Backend (FastAPI)
           │
           ├── featurize() extracts 7-feature behavioral vector per action
           ├── IsolationForest scores each action in real time
           └── Redis + Neo4j persist the full attack session
           │
           ▼
     This dataset ← exported from backend after attack session
```

All attacks were conducted in an isolated lab environment against self-owned Docker honeypots.  
**Source code:** [github.com/Deaker-09/ShadowMess-AI](https://github.com/Deaker-09/ShadowMess-AI)